In [5]:
import sys
sys.path.append("..")
import pandas as pd
from src.cleaning import clean_patients, clean_encounters

patients = clean_patients(pd.read_csv("../data/raw/patients.csv"))

encounters = clean_encounters(pd.read_csv("../data/raw/encounters.csv"))
encounters.info()

# one to many relationship
encounters["PATIENT"].nunique()

# Count encounters per patient
encounters_per_patient = (encounters
                         .groupby("PATIENT")
                         .size()
                         .sort_values(ascending=False)
                         )
encounters_per_patient.head()


<class 'pandas.DataFrame'>
RangeIndex: 5473 entries, 0 to 5472
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   Id                   5473 non-null   str                
 1   START                5473 non-null   datetime64[us, UTC]
 2   STOP                 5473 non-null   datetime64[us, UTC]
 3   PATIENT              5473 non-null   str                
 4   ORGANIZATION         5473 non-null   str                
 5   PROVIDER             5473 non-null   str                
 6   PAYER                5473 non-null   str                
 7   ENCOUNTERCLASS       5473 non-null   str                
 8   CODE                 5473 non-null   int64              
 9   DESCRIPTION          5473 non-null   str                
 10  BASE_ENCOUNTER_COST  5473 non-null   float64            
 11  TOTAL_CLAIM_COST     5473 non-null   float64            
 12  PAYER_COVERAGE       5473 non-n

PATIENT
688c8453-ba6f-7dec-03c3-eaa27d6df1a4    607
7ad140ab-bfae-c3ae-20a3-2244b1c4d0e2    524
5d84e6a3-b4bd-63d6-57c5-cada0916490d    333
21060120-8398-1940-3dc7-1d7f56bb2674    144
8d7f6a31-31ba-da9c-2b57-03ee0f7577a0    121
dtype: int64

In [6]:

# Calculate encounter duration (length of stay)
encounters["duration_hours"] = (encounters["STOP"] - encounters["START"]).dt.total_seconds() / 3600
print(encounters["duration_hours"].describe())

# What is the earliest encounter date?
# What is the latest encounter date?
print("earliest:", encounters["START"].min())
print("latest  :", encounters["START"].max())
print("span    :", encounters["START"].max() - encounters["START"].min())

#  encounters per year and per month
encounters_per_year = encounters["START"].dt.year.value_counts().sort_index()
print(encounters_per_year.head(10))

encounters_per_month = encounters["START"].dt.month.value_counts().sort_index()
print(encounters_per_month)


count    5473.000000
mean        4.893613
std        52.933138
min         0.250000
25%         0.250000
50%         0.698056
75%         1.482222
max      1632.250000
Name: duration_hours, dtype: float64
earliest: 1952-10-22 23:17:16+00:00
latest  : 2026-07-15 01:29:39+00:00
span    : 26928 days 02:12:23
START
1952    1
1957    3
1960    2
1961    1
1963    1
1964    3
1968    1
1969    4
1970    2
1971    2
Name: count, dtype: int64
START
1     425
2     432
3     449
4     497
5     544
6     480
7     493
8     423
9     392
10    409
11    465
12    464
Name: count, dtype: int64


In [7]:
# Merge patients and encounters
merged = encounters.merge(
    patients[["Id", "FIRST", "LAST", "GENDER", "age", "age_group"]],
    left_on="PATIENT",
    right_on="Id",
    how="left"
)
merged.head()

,Id_x,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,...,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,duration_hours,Id_y,FIRST,LAST,GENDER,age,age_group
0,f9ba83f7-9940-16ae-5f88-aea1af242f71,1973-02-01 07:34:23+00:00,1973-02-01 07:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,497f39dd-280e-3d58-af5b-c5e3a3a09b10,2352c51f-5dce-383b-adbf-bf5cc1f1c4d7,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185347001,Encounter for problem (procedure),...,0.00,256355007.0,Glycine max (substance),0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.8,middleage
1,a1733070-046a-4506-13b1-f518d757cdd0,1997-04-22 16:12:20+00:00,1997-04-22 16:59:47+00:00,a1733070-046a-4506-bba6-47f32652e9d7,c64c848c-0315-3115-87fa-80c360bb6a73,35b13cfd-13a0-34e4-a83d-32af4b12344a,df166300-5a78-3502-a46a-832842197811,wellness,162673000,General examination of patient (procedure),...,554.20,NaN,NaN,0.790833,a1733070-046a-4506-bba6-47f32652e9d7,Donte636,Daugherty69,M,47.5,middleage
2,f9ba83f7-9940-16ae-81d1-8cdb486ca3ad,1973-02-16 21:34:23+00:00,1973-02-16 21:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,497f39dd-280e-3d58-af5b-c5e3a3a09b10,2352c51f-5dce-383b-adbf-bf5cc1f1c4d7,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185347001,Encounter for problem (procedure),...,0.00,609328004.0,Allergic disposition (finding),0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.8,middleage
3,f9ba83f7-9940-16ae-158c-5cd0e5a140b9,1989-01-10 22:34:23+00:00,1989-01-10 22:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,f22bc2ac-f8bd-3f9f-9a40-93db37639d3f,b13a2be9-5dba-3041-b1c6-62ff62c512ea,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,410620009,Well child visit (procedure),...,0.00,NaN,NaN,0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.8,middleage
4,a1733070-046a-4506-0d73-cf2822965f28,2013-05-14 16:12:20+00:00,2013-05-14 17:09:30+00:00,a1733070-046a-4506-bba6-47f32652e9d7,c64c848c-0315-3115-87fa-80c360bb6a73,35b13cfd-13a0-34e4-a83d-32af4b12344a,df166300-5a78-3502-a46a-832842197811,wellness,162673000,General examination of patient (procedure),...,1616.16,NaN,NaN,0.952778,a1733070-046a-4506-bba6-47f32652e9d7,Donte636,Daugherty69,M,47.5,middleage


In [8]:
# encounter types
encounter_summary = pd.DataFrame({
    "count": encounters["ENCOUNTERCLASS"].value_counts(),
    "percent": (encounters["ENCOUNTERCLASS"].value_counts(normalize=True) * 100).round(1),
})
encounter_summary


,count,percent
ENCOUNTERCLASS,,
ambulatory,2781,50.8
wellness,1344,24.6
outpatient,745,13.6
urgentcare,270,4.9
emergency,191,3.5
home,62,1.1
inpatient,55,1.0
snf,12,0.2
hospice,9,0.2
